# SI Figs. S79 and S80 — quantifying dye-pair unmixing (Ximea)

**Makes up:** SI Fig. S79 b)–d), the workflow of the misidentification analysis for the ATTO 565 + ATTO 633
mixture, and SI Fig. S80, the four mixtures sorted at one photoelectron level each. Panel a) of S79 (the flow
diagram) is drawn by hand.

**Data and analysis:** exactly those of Figure 3 a–c (`Figure_3a-c_DyePairs.ipynb`; BioStudies `S-BIAD3210`,
`2026_Multicolour_Paper/Ximea/DyeMixtures/`, a 4 × 4 grid of positions per pair). This notebook uses the same
output folder, so the downloads, fits and molecules of Figure 3 a–c are reused (or made here if absent), and the
misidentification analysis is the one behind Figure 3c (`dp.misidentification_analysis`). Every rate shown is a
point of Figure 3c.

**Photoelectrons, not `photons`:** the published panels binned on `photons` (A_B + A_G + A_R), which overcounts
detected photoelectrons: on the Bayer mosaic, photoelectrons = `photons` × 0.25 (1 + A_G), i.e. × 0.43 for
ATTO 520 + ATTO Rho6G and × 0.32–0.33 for the other pairs (mean A_G of the pair's two dyes). Each published range is
converted with its pair's factor and the bin of Figure 3c's bins (`dp.PE_BINS`) that holds the converted mean is
shown:

| panel | published (`photons`) | converted | bin shown (photoelectrons) |
|---|---|---|---|
| S79 b) reference molecules | > 20,000 | > 6,600 | > 6,600 (`dp.REFERENCE_CUTS`) |
| S79 c), d) ATTO 565 + ATTO 633 | 1,760–2,508 | 580–830 | 562–750 |
| S80 a) ATTO 520 + ATTO Rho6G | mean 5,737 | 2,470 | 2,371–3,162 |
| S80 b)–d) other pairs | mean ~5,000 | 1,570–1,660 | 1,334–1,778 |

---
## 0 — Imports

In [ ]:
import sys
import types
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# _biostudies_download.py, _multifov.py and _dye_pairs.py live one level up (notebooks/figures/).
sys.path.insert(0, str(Path.cwd().resolve().parent))
import _dye_pairs as dp

from pyS3M import IOFunctions
from pyS3M import sCMOSFunctions
from pyS3M import SR_Functions
from pyS3M import SM_extractionfunctions
from pyS3M import PlottingBase
from pyS3M.DriftCorrectionFunctions import Drift_Correction_Functions

IO = IOFunctions.IO_Functions()
sCMOS = sCMOSFunctions.sCMOS_Functions()
SupRes_F = SR_Functions.SuperRes_Functions()   # Ximea defaults (69 nm pixels)
SM_E = SM_extractionfunctions.extract_SMs()
drift_corrector = Drift_Correction_Functions()
plotter = PlottingBase.PublicationPlotter(dark_background=False)

print("Python:", sys.version)

---
## 1 — Configuration

As Figure 3 a–c, with the same output folder (`OUT_ROOT`), so its downloads, fits and molecules are reused.

In [ ]:
FAST_MODE = True   # True: one position per pair (~8 GB). False: all 64 positions (~130 GB, the published data).
REFIT = False      # True: fit again even if fits with the same files and settings exist.

REPO_ROOT = Path.cwd().resolve().parents[2]   # notebooks/figures/SI -> repo root
CALIB_DIR = REPO_ROOT / "Camera_Calibrations" / "Ximea_Camera"
OUT_ROOT  = Path("..") / "outputs" / "Figure_3a-c_DyePairs"   # Figure 3 a-c's folder (git-ignored)
FIG_DIR   = OUT_ROOT / ("fast" if FAST_MODE else "full")
# Point DATA_ROOT at a drive with room for the download. Each pair gets its own sub-folder,
# separately for each FAST_MODE setting: the fit reads every TIFF inside it.
DATA_ROOT = OUT_ROOT / ("data_fast" if FAST_MODE else "data_full")
FIG_DIR.mkdir(parents=True, exist_ok=True)
DATA_ROOT.mkdir(parents=True, exist_ok=True)

calib = dict(
    gain_map=IO.read_tiff(CALIB_DIR / "gain.tif"),
    offset_map=IO.read_tiff(CALIB_DIR / "offset.tif"),
    rqe=IO.read_tiff(CALIB_DIR / "rqe.tif"),
    read_noise=IO.read_tiff(CALIB_DIR / "readnoise.tif"),
    variance=IO.read_tiff(CALIB_DIR / "variance.tif"),
)

smoothing_function = types.SimpleNamespace()
smoothing_function.args = {"sigma": 1.5}
smoothing_function.extent = 1.5
smoothing_function.smoothing_function = sCMOS.gaussian_filter_stack
smoothing_function.data_arg = "image"

PAIRS = list(dp.PAIRS)
print(f"FAST_MODE  : {FAST_MODE}")
print(f"Data dir   : {DATA_ROOT.resolve()}")
print(f"Figures dir: {FIG_DIR.resolve()}")
print("Fit settings:", dp.FIT_PARAMS, "(PSF wavelength per pair)")

---
## 2 — Download, fit, drift-correct and cluster each pair

As Figure 3 a–c (sections 2 and 3): reused when Figure 3 a–c has run with the same `FAST_MODE`.

In [ ]:
pair_files = {pair: dp.prepare_acquisition(pair, DATA_ROOT / pair, FAST_MODE, SupRes_F, calib,
                                           smoothing_function, drift_corrector, IO, refit=REFIT)
              for pair in PAIRS}
for pair, files in pair_files.items():
    print(f"{pair}: {len(files)} position(s)")

In [ ]:
sm_db, sf_db, pa_db = {}, {}, {}
for pair in PAIRS:
    sm, sf = dp.cluster_sample(pair, SM_E, pair_files[pair], DATA_ROOT / pair)
    sm_db[pair], sf_db[pair] = IO.add_photoelectron_columns(sm), IO.add_photoelectron_columns(sf)
    pa_db[pair] = dp.photoelectron_accumulation(sf_db[pair], IO)
    print(f"{pair}: {len(sm):,} molecules, {len(sf):,} localisations, "
          f"median {sm_db[pair]['photoelectrons'].median():,.0f} photoelectrons per molecule")

---
## 3 — Misidentification analysis (Figure 3c)

Per pair: the two mean colours from a 2-component GMM on the reference molecules (`dp.REFERENCE_CUTS`); then, in
each bin of accumulated photoelectrons, each reference molecule's colour at its first entry into the bin, the two
covariances and weights fitted with the means fixed (Tukey M-estimator), and the misidentification rate from the
two Gaussians' overlap (Monte Carlo, 10⁵ samples). The molecules of a bin are sorted by Bayes' decision rule with
that bin's model.

In [ ]:
analysis = {pair: dp.misidentification_analysis(pair, sm_db[pair], pa_db[pair], SM_E) for pair in PAIRS}
for pair, a in analysis.items():
    print(f"{pair}: {len(a['ref_db']):,} reference molecules; " + "; ".join(
        f"{dp._dye_name(pair, k)} A_R {m[0]:.3f} A_G {m[1]:.3f}" for k, m in enumerate(a["means"])))

---
## 4 — SI Fig. S79 b)–d): the workflow, ATTO 565 + ATTO 633

b) The reference molecules (> 6,600 photoelectrons, in the pair's colour box): the 2-component GMM's means, with
the widths and weights refitted with those means fixed (Tukey M-estimator, as in c) and Figure 3c's bins). The GMM's
own widths are inflated by the ~4 % of reference molecules that are two overlapping emitters (intermediate colour
in every frame, wider PSF) or two molecules merged in time;
c) the molecules of one bin (562–750 accumulated photoelectrons) with the components refitted, means fixed;
d) the same molecules sorted by Bayes' rule. Left: pixel 3 (A_R); right: pixel 2 (A_G).

In [ ]:
a = analysis[dp.WORKFLOW_PAIR]
w = dp.bin_example(a, pa_db[dp.WORKFLOW_PAIR], dp.WORKFLOW_BIN)
fig, axs, note = dp.plot_workflow(a["ref_db"], a["ref_weights_robust"], a["means"], a["ref_covs_robust"], w["mols"],
                                  w["covs"], w["weights"],
                                  (w["row"]["photoelectron_bin_min"], w["row"]["photoelectron_bin_max"]))
fig.savefig(FIG_DIR / "FigureSI_DyePairWorkflow.svg", dpi=600, format="svg")
plt.show()
print(note)
print(f"Misidentification in this bin: {100 * w['error_rate']:.2g} %; "
      f"sorted: {np.bincount(w['labels'], minlength=2).tolist()}")

---
## 5 — SI Fig. S80: the four pairs at one photoelectron level

The pixel that separates each pair (pixel 2 for ATTO 520 + ATTO Rho6G, pixel 3 for the others) for the molecules of
the pair's bin (`dp.EXAMPLE_BINS`, table above), sorted by Bayes' rule; the bin's misidentification rate (Figure
3c's point) and the molecules' mean accumulated photoelectrons.

In [ ]:
examples = {pair: dp.bin_example(analysis[pair], pa_db[pair], lo) for pair, lo in dp.EXAMPLE_BINS.items()}
fig, axs = dp.plot_examples(examples)
fig.savefig(FIG_DIR / "FigureSI_DyePairExamples.svg", dpi=600, format="svg")
plt.show()
print(pd.DataFrame({dp._dye_name(p, 0) + " + " + dp._dye_name(p, 1): {
    "bin (pe)": f"{e['row']['photoelectron_bin_min']:,.0f}–{e['row']['photoelectron_bin_max']:,.0f}",
    "molecules": len(e["mols"]),
    "mean pe": round(e["mols"]["photoelectrons_accumulated"].mean()),
    "% misidentified": f"{100 * e['error_rate']:.2g}"} for p, e in examples.items()}).T)